# Production-ноутбук: BEN-фреймворк для анализа разнообразия рекомендаций

**Версия:** `v11_unified_permutation_with_unique_rel_metrics`

Считает 10 метрик разнообразия выдачи (BEN, BEN_weighted, 4 unique абсолютных, 4 unique относительных) для пар control vs treatment через единый permutation-тест на уровне пользователей. AA-валидация на реальных Fashion-данных подтвердила корректную калибровку p-value (Type I error 4.5–6.0%, все 5 тестов калибровки пройдены для всех 10 метрик).

**Главные особенности:**
- HDFS-кэш материализации (быстрый resume после падения)
- Чекпоинт результатов после каждой пары
- Auto-recovery Spark session (auto-recreate при kill YARN)
- BLAS-thread limit для совместимости с multiprocessing
- 230 нс/строка/permutation на NUMA-bound железе

**Структура:**
1. Spark setup и helpers
2. Параметры эксперимента
3. Микрокатегории (фильтр по логкату)
4. Участники эксперимента + семплинг
5. Сессии → impressions с action-флагами
6. Join + дедуп + материализация в pandas
7. Unified permutation-тест (главное ядро)
8. analyze_pair (оркестратор пары)
9. Главный цикл по парам (с resume и resilience)
10. Сводный отчёт
11. Сохранение результатов в HDFS


## Заголовок и changelog

```
# =============================================================================
# PySpark ноутбук: BEN-анализ v11 — оптимизированный + *_rel метрики
# =============================================================================
#
# v11 ИЗМЕНЕНИЯ ОТ v10:
#   1. Оптимизирована функция honest_user_permutation_test:
#      - unique_items вычисляется через (c_A>0).sum() — БЕСПЛАТНО
#        (c_A/c_B уже вычисляется для BEN)
#      - action-subsets (clicks/btc/contacts) вынесены за loop
#      - 1/k_u, 1/sum_w_user — pre-compute
#      - nlq_per_row через `nlq_B[i_arr]` + in-place правка для A
#        (экономит ~400MB памяти на 25M строк)
#   2. ДОБАВЛЕНО 4 новые метрики *_rel:
#      - unique_items_rel, unique_clicked_items_rel,
#        unique_btc_items_rel, unique_contacted_items_rel
#      Это (T-C)/C на каждой permutation, интерпретируемее в %.
#      ВАЖНО: corr(p_abs, p_rel) ≈ 0.998 — это та же информация
#      в более интерпретируемой шкале, не доп. статсила.
#   3. Resilience:
#      - HDFS-кэш материализации pair_df (cache_path в honest_user_permutation_test)
#      - Чекпоинт результата каждой пары — при перезапуске пропускаем
#        уже посчитанные пары
#      - Auto-recovery Spark session (ensure_spark_alive)
#      - OPENBLAS_NUM_THREADS=1 для совместимости с multiprocessing
#
# МЕТОД ДЛЯ BEN (v11): Honest user-level perm + q_per_group
#   В одном permutation loop считаются 10 метрик (см. ниже).
#   Bucket-perm + q_global был отдельным sanity-методом в v10, в v11
#   убран: AA-валидация подтвердила корректность honest user perm,
#   дополнительный sanity-check не нужен.
#
# UNIFIED PERMUTATION metrics (10 за один loop):
#   Абсолютные:
#     - BEN, BEN_weighted
#     - UniqueItems, UniqueClickedItems, UniqueBTCItems, UniqueContactedItems
#   Относительные (NEW v11):
#     - UniqueItems_rel, UniqueClickedItems_rel,
#       UniqueBTCItems_rel, UniqueContactedItems_rel
#
# =============================================================================
# AA-ВАЛИДАЦИЯ v11 НА РЕАЛЬНЫХ Fashion-ДАННЫХ — ВСЕ МЕТРИКИ КАЛИБРОВАНЫ ✓
# =============================================================================
#   Параметры:    200 AA-итераций × 150 perm, 3% sample (~13M строк после dedup)
#   5 тестов калибровки на каждой метрике:
#     [1] α∈Wilson   — 95% Wilson CI на эмпирическую Type I error содержит 5%
#     [2] α∈CP       — то же через Clopper-Pearson exact CI
#     [3] Binom test — p-value двустороннего биномиального теста H0: p=α > 0.05
#     [4] KS-uniform — KS-test на Uniform(0,1) не отвергает (p>0.05)
#     [5] AD-uniform — Anderson-Darling на хвосты не отвергает
#
#   ВСЕ 10 МЕТРИК прошли ВСЕ 5 ТЕСТОВ ✓
#
#   ┌──────────────────────────────┬─────────┬─────────────┬──────────┐
#   │ Метрика                      │ α_emp   │ Wilson CI   │ Verdict  │
#   ├──────────────────────────────┼─────────┼─────────────┼──────────┤
#   │ ben                          │  4.5%   │ [2.4, 8.3]  │ ✓ OK     │
#   │ ben_weighted                 │  5.5%   │ [3.1, 9.6]  │ ✓ OK     │
#   │ unique_items                 │  4.5%   │ [2.4, 8.3]  │ ✓ OK     │
#   │ unique_clicked_items         │  5.5%   │ [3.1, 9.6]  │ ✓ OK     │
#   │ unique_btc_items             │  6.0%   │ [3.5,10.2]  │ ✓ OK     │
#   │ unique_contacted_items       │  4.5%   │ [2.4, 8.3]  │ ✓ OK     │
#   │ unique_items_rel             │  4.5%   │ [2.4, 8.3]  │ ✓ OK     │
#   │ unique_clicked_items_rel     │  5.5%   │ [3.1, 9.6]  │ ✓ OK     │
#   │ unique_btc_items_rel         │  6.0%   │ [3.5,10.2]  │ ✓ OK     │
#   │ unique_contacted_items_rel   │  4.5%   │ [2.4, 8.3]  │ ✓ OK     │
#   └──────────────────────────────┴─────────┴─────────────┴──────────┘
#
#   Все Binom p > 0.5 — гипотеза «Type I = α» НЕ отвергается ни для одной метрики.
#   Все KS-stat ≤ 0.087 при D-крит=0.096 — uniformity подтверждена.
#   Все AD-stat ≤ 1.81 при крит=2.49 — нет AD-tail артефактов.
#   Под H0 mean(Δ) ≪ std(Δ) для всех — bias отсутствует.
#
#   Вывод: метод корректен, можно ставить в прод без дополнительных sanity-проверок.
#
# =============================================================================
# БЕНЧМАРК v11 СКОРОСТИ:
#   - На синтетике (14.8M rows × 30 perm): ~83 ns/row/perm
#   - На реальных Fashion-данных (NUMA-bound): ~230 ns/row/perm
#
# ПРОГНОЗ ДЛИТЕЛЬНОСТИ v11 (3 пары × 500 perm, sequential):
#   ┌──────────┬─────────────────┐
#   │ Rows pair│ Real-data time  │
#   ├──────────┼─────────────────┤
#   │  ~13M    │ ~50 мин         │  ← 3% sample
#   │  ~22M    │ ~1.5 ч          │  ← 5% sample
#   │  ~45M    │ ~3 ч            │  ← 10% sample
#   └──────────┴─────────────────┘
#
# При падении (YARN-kill, kernel-restart) — Run All подхватит из чекпоинтов.
# KEEP-ALIVE PING Spark каждые PING_EVERY_ITERS итераций внутри циклов.
# =============================================================================
```

## Ячейка 1 — Инициализация Spark, BLAS-thread limit и resilient session

**Что происходит:**
1. **Жёстко ограничиваем BLAS-threads до 1** (`OPENBLAS_NUM_THREADS`, `OMP_NUM_THREADS`, и т.д.) **до** `import numpy`. Это критично для multiprocessing-окружения: иначе OpenBLAS по дефолту хапает столько threads сколько CPU видит OS (96 на твоей машине), и при работе с pandas-loop через несколько workers получаем oversubscription с замедлением 8–10×.
2. **Создаём resilient Spark session** через `get_or_create_spark()` — она умеет пересоздаваться с retry+backoff если YARN убил предыдущую. Дополнительные helpers:
   - `spark_ping()` — лёгкий keep-alive
   - `ensure_spark_alive()` — проверяет жива ли session, пересоздаёт при необходимости
   - `_hdfs_path_exists()` / `_hdfs_delete()` — работа с HDFS через Hadoop FileSystem API

**Зачем resilient session:** долгие прогоны (1–4 часа) часто прерываются убийством Spark application'а со стороны YARN. Без auto-recovery ты теряешь весь прогресс. С ним — `Run All` после kernel restart подхватит работу с точки чекпоинта.

**Особенности конфига Spark:**
- `executor.cores=4, executor.memory=12g, maxExecutors=200` — для широких pandas-операций с большими impressions
- `arrow.pyspark.enabled=true` — ускоряет `toPandas()` через Arrow IPC формат
- `kryoserializer.buffer.max=1g` — нужно для больших broadcast'ов микрокатегорий


In [ ]:

# КРИТИЧНО: ограничить BLAS-threads ДО import numpy (см. AA-notebook).
# Multiprocessing-параллелизм + дефолтный OpenBLAS = oversubscription и 8-10× замедление.
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["MKL_NUM_THREADS"]      = "1"
os.environ["NUMEXPR_NUM_THREADS"]  = "1"

import time, json, hashlib
from datetime import date, timedelta
from functools import reduce

import numpy as np
import pandas as pd

import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import SparkSession

from search_quality_dags.jupyter import spark_task

spark_cfg = {
    "spark.executor.cores": 4,
    "spark.executor.memory": "12g",
    "spark.dynamicAllocation.maxExecutors": 200,
    "spark.driver.memory": "64g",
    "spark.driver.maxResultSize": "32g",
    "spark.sql.shuffle.partitions": "2000",
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.orc.filterPushdown": "true",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.kryoserializer.buffer.max": "1g",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.sql.execution.arrow.pyspark.fallback.enabled": "true",
    "spark.sql.execution.arrow.maxRecordsPerBatch": "50000",
    "spark.hadoop.net.topology.node.switch.mapping.impl":
        "org.apache.hadoop.net.TableMapping",
}

# === Resilient Spark session ===
_SPARK_TASK = None
_SPARK_APP_NAME = "avchuykov ben-v11"


def get_or_create_spark(force_new=False):
    """Получить рабочую SparkSession, при потере — пересоздать.
    Возвращает (spark, sc, was_recreated)."""
    global _SPARK_TASK
    needs_create = force_new or _SPARK_TASK is None
    if not needs_create:
        try:
            sc = _SPARK_TASK.spark_session.sparkContext
            if sc._jsc is None:
                needs_create = True
            else:
                _ = sc.parallelize([1]).count()
        except Exception as e:
            print(f"  [resilient] Existing session dead: {type(e).__name__}: {e}")
            needs_create = True
    
    if needs_create:
        action = "Recreating" if _SPARK_TASK is not None else "Creating"
        print(f"  [resilient] {action} Spark session...", flush=True)
        if _SPARK_TASK is not None:
            try: _SPARK_TASK.spark_session.stop()
            except Exception: pass
        last_exc = None
        for attempt in range(3):
            try:
                _SPARK_TASK = spark_task(app_name=_SPARK_APP_NAME, cfg=spark_cfg)
                spark = _SPARK_TASK.spark_session
                spark.sparkContext.setLogLevel("ERROR")
                log4j = spark._jvm.org.apache.log4j
                log4j.LogManager.getLogger("org.apache.hadoop.net.ScriptBasedMapping").setLevel(log4j.Level.ERROR)
                log4j.LogManager.getLogger("org.apache.hadoop.util.Shell").setLevel(log4j.Level.ERROR)
                log4j.LogManager.getLogger("org.apache.hadoop.net").setLevel(log4j.Level.ERROR)
                import logging
                logging.getLogger("py4j").setLevel(logging.ERROR)
                logging.getLogger("py4j.java_gateway").setLevel(logging.ERROR)
                print(f"  [resilient] Session ready (attempt {attempt+1}): {spark.version}", flush=True)
                return spark, spark.sparkContext, True
            except Exception as e:
                last_exc = e
                print(f"  [resilient] attempt {attempt+1}/3 failed: {type(e).__name__}: {e}")
                time.sleep(min(30 * 2**attempt, 120))
        raise RuntimeError(f"Не удалось создать Spark session за 3 попытки: {last_exc}")
    
    return _SPARK_TASK.spark_session, _SPARK_TASK.spark_session.sparkContext, False


def spark_ping(silent=False):
    """Keep-alive ping. Возвращает True если жива, False если нет."""
    global _SPARK_TASK
    if _SPARK_TASK is None: return False
    try:
        _ = _SPARK_TASK.spark_session.sparkContext.parallelize([1, 2, 3]).count()
        if not silent: print(f"    [spark ping OK]", flush=True)
        return True
    except Exception as e:
        if not silent: print(f"    [spark ping FAILED: {type(e).__name__}: {e}]", flush=True)
        return False


def ensure_spark_alive():
    """Если session мертва — пересоздать. Возвращает (spark, was_recreated)."""
    global _SPARK_TASK
    if not spark_ping(silent=True):
        print(f"    [resilient] Session dead, recreating...", flush=True)
        spark, sc, recreated = get_or_create_spark(force_new=True)
        return spark, True
    return _SPARK_TASK.spark_session, False


def _hdfs_path_exists(path):
    try:
        jvm = _SPARK_TASK.spark_session._jvm
        hadoop_path = jvm.org.apache.hadoop.fs.Path(path)
        fs = jvm.org.apache.hadoop.fs.FileSystem.get(
            jvm.java.net.URI.create(path),
            _SPARK_TASK.spark_session._jsc.hadoopConfiguration()
        )
        return fs.exists(hadoop_path)
    except Exception:
        return False


def _hdfs_delete(path):
    try:
        jvm = _SPARK_TASK.spark_session._jvm
        hadoop_path = jvm.org.apache.hadoop.fs.Path(path)
        fs = jvm.org.apache.hadoop.fs.FileSystem.get(
            jvm.java.net.URI.create(path),
            _SPARK_TASK.spark_session._jsc.hadoopConfiguration()
        )
        if fs.exists(hadoop_path):
            fs.delete(hadoop_path, True)
    except Exception as e:
        print(f"  [hdfs_delete] {e}")


# Создаём session
spark, sc, _ = get_or_create_spark()
print(f"Spark: {spark.version}")

## Ячейка 2 — Параметры эксперимента

**Что нужно проверять при запуске:**
- `EXPERIMENT_ID` — id эксперимента в Trisigma
- `DATE_FROM` / `DATE_TO` — окно анализа (включая обе границы)
- `CONTROL_LABEL` / `TREATMENT_LABELS` — имена групп точно как в `ab_participant_extended`
- `SAMPLE_PCT` — % семплирования по `cookie_id`. Хеш детерминированный (`SAMPLE_SALT`), при перезапуске со теми же параметрами получишь тех же пользователей
- `LOGCATS` — фильтр по логкатегории. **Пустой список** = без фильтра (тотал)
- `N_PERMUTATIONS_HONEST = 500` — количество перестановок для permutation-теста
- `COMPUTE_CI` / `CI_METRICS` — считать ли H₀-границы для каждой метрики

**Важный момент про пути HDFS:** в путь кэша и результатов включён тег `_logcats_tag` — это означает что прогон с `LOGCATS=["Goods.Fashion"]` и `LOGCATS=[]` пишут в **разные директории** и не путают друг друга. Если поменяешь логкаты — старые результаты сохранятся, новый прогон стартует с чистого листа.

**Прогноз времени** в конце ячейки — оценивает на основе эмпирической константы 230 нс/строка/permutation, замеренной на NUMA-bound железе твоего driver. Считает ожидаемое время 3 пар × N permutations.


In [ ]:

# Источники в HDFS
PATH_PARTICIPANTS = "/trino/ab_participant_extended"
PATH_SESSIONS     = "/user/airflow/user_sessions"

# Период эксперимента
DATE_FROM = "2026-03-13"
DATE_TO   = "2026-03-18"
EXPERIMENT_ID = 30895

# Группы
CONTROL_LABEL    = "control"
TREATMENT_LABELS = ["v1i_personal", "v1j_personal", "v1k_personal"]

# Подвыборка юзеров
SAMPLE_PCT  = 10
SAMPLE_SALT = "exp30895_sample_v1"

# Top-K: ограничение выдачи по позиции
TOP_K = 100

# Параметры тестов
SEED           = 52
ALPHA          = 0.05

# Поправка на множественные сравнения
# "none" | "bonferroni"
MULTIPLE_CORRECTION = "none"

# Exposure-фильтр
USE_EXPOSURE_FILTER = False

# === v7 параметры ===

# Фильтр по logical_category (в 30895 treatment меняет только Fashion)
PATH_MICROCATEGORIES = "/vertica/ext/current_microcategories"
LOGCATS = ["Goods.Fashion"]             # [] = без фильтра

# Honest user-level permutation test для BEN (MAIN inference, q_per_group).
# AA-тест: bucket+q_per_group даёт α=0.38, а honest+q_per_group = 0.06 ✓
# БЕНЧМАРК: ~85 нс/строку/итерацию.
# На 200M строк 500 perm × 3 пары ≈ 7 часов.
# Если много строк — уменьши до 200 (точность p-value ±0.02).
N_PERMUTATIONS_HONEST = 500

# CI на основе квантилей permutation distribution (под H0).
# Эффект значим если delta_obs выходит за квантили [α/2, 1-α/2] этого распределения.
# COMPUTE_CI: True | False
# CI_METRICS:
#   "all" — для всех 6 метрик (BEN, BEN_weighted, 4 unique)
#   None  — не считаем (только p-value)
#   список (например ["unique_items"]) — только для указанных
COMPUTE_CI = True
CI_METRICS = "all"

# Keep-alive пинг Spark (чтобы YARN не убил AM).
PING_EVERY_ITERS = 50

# Путь для результатов
HDFS_HOME = "/user/avchuykov"
_mode = "exposure" if USE_EXPOSURE_FILTER else "assigned"
# Тег логкатов в путь — чтобы прогоны с разными LOGCATS не делили один кэш.
# Пустой список → "all", иначе сортируем и склеиваем имена через "+".
_logcats_tag = (
    "all" if not LOGCATS
    else "+".join(sorted(LOGCATS)).replace(".", "_").replace("/", "_").replace(" ", "_")
)
PATH_RESULTS = f"{HDFS_HOME}/ben_results/exp{EXPERIMENT_ID}/topk_{TOP_K}_{_mode}_lc-{_logcats_tag}_v11"

print(f"\n=== exp{EXPERIMENT_ID} v11 (unified permutation + 4 *_rel + cache + resilience) ===")
print(f"  Период:        {DATE_FROM}..{DATE_TO}")
print(f"  Treatments:    {TREATMENT_LABELS}")
print(f"  Logcats:       {LOGCATS}")
print(f"  Sample:        {SAMPLE_PCT}% (salt={SAMPLE_SALT})")
print(f"  TOP_K:         {TOP_K}")
print(f"  Honest perm:   {N_PERMUTATIONS_HONEST} (q_per_group — BEN + 4 unique + 4 unique_rel)")
print(f"  CI:            compute={COMPUTE_CI}, metrics={CI_METRICS}")
print(f"  α:             {ALPHA}, correction: {MULTIPLE_CORRECTION}")
print(f"  Ping каждые:   {PING_EVERY_ITERS} iter")
print(f"  Results path:  {PATH_RESULTS}")

## Ячейка 3 — Загрузка микрокатегорий для фильтра по логкатегории

**Что происходит:** если задан `LOGCATS`, читаем витрину `current_microcategories` и фильтруем `microcat_id` которые попадают в нужную логкатегорию. Затем в Ячейке 6 эта таблица будет заджойнена с impressions, чтобы оставить только показы с нужной логкатегорией.

**Особенности:**
- Витрина `microcategory` содержит дерево: `microcat_id → logcat_id → vertical_id`
- Один логкат содержит много микрокатегорий, поэтому фильтрация на уровне `microcat_id` корректнее чем по строке логката
- `broadcast()` над таблицей микрокатегорий — она маленькая, ~10–50K строк, broadcast-join намного быстрее shuffle-join

Если `LOGCATS=[]` — ячейка пропускается, идём по тоталу.


In [ ]:

if LOGCATS:
    t = time.time()
    mcats_df = (
        spark.read.parquet(PATH_MICROCATEGORIES)
            .filter((F.col("is_active_microcat") == True) &
                     (F.col("logical_category").isin(*LOGCATS)))
            .select(F.col("microcat_ext").cast("long").alias("microcat_ext"),
                     F.col("logical_category"))
            .distinct()
    )
    mcats_df.cache()
    n_mcats = mcats_df.count()
    print(f"Microcategories: {n_mcats:,} за {time.time()-t:.0f}s")
    if n_mcats == 0:
        raise ValueError(f"Нет микрокатегорий для {LOGCATS}")
else:
    mcats_df = None
    print("LOGCATS=[] — фильтр по logcat не применяется")

## Ячейка 4 — Участники эксперимента + семплирование

**Что происходит:**
1. Из `ab_participant_extended` достаём всех `cookie_id` относящихся к `EXPERIMENT_ID` с указанной группой (`control` + 3 treatment).
2. Применяем **детерминированный hash-семплинг**: `abs(hash(cookie_id || SALT)) % 100 < SAMPLE_PCT`. Это даёт `SAMPLE_PCT %` пользователей при детерминированном выборе — при повторных запусках получаем **тех же** пользователей.
3. Бакетируем юзеров через `bucket_uid = abs(hash(cookie_id || BUCKET_SALT)) % 1024` — нужно для возможной параллелизации внутри permutation-loop, плюс бакет-id используется при некоторых стратификациях.

**Зачем именно hash-семплинг, а не `sample()`:**
- Воспроизводимость: тот же seed = те же люди в выборке
- Можно сравнивать прогоны с одинаковыми параметрами, не боясь что выборка случайно поменялась
- Безопасный resume: если упало после семплирования, при перезапуске семплируем тех же

**Размер выборки:** при `SAMPLE_PCT=10` это ~5M пользователей × 4 группы = ~1.29M на группу.


In [ ]:
#
# Колонки в ab_participant_extended:
#   participant_id, participant_ext_id, variant, bucket (0-255),
#   is_exposed, event_date, experiment_id
#
# ВАЖНО про bucket:
#   bucket это число 0-255 в каждой группе. Значит bucket=42 встречается в
#   control, v1i, v1j, v1k — это 4 разных рандомизированных группы юзеров.
#   Для bucket-permutation test нужен УНИКАЛЬНЫЙ id бакета между группами:
#   bucket_uid = concat(ab_group, bucket) → "control_42", "v1i_personal_42" ...
#   Иначе permutation test схлопнет 1024 бакета в 256 и развалится.
#
# Шаги:
#   1. Фильтр по event_date (partition pruning), experiment_id, variant
#   2. (опц.) is_exposed=True
#   3. groupBy(participant_id, participant_ext_id) → одна строка на юзера
#      с min(event_date) как first_exposure_date и first(variant/bucket)
#   4. bucket_uid = "{ab_group}_{bucket}"
#   5. Хеш-подвыборка по cookie_id (= participant_id)

t = time.time()

base = (
    spark.read.parquet(PATH_PARTICIPANTS)
        .filter(F.col("event_date").between(DATE_FROM, DATE_TO))
        .filter(F.col("experiment_id") == EXPERIMENT_ID)
        .filter(F.col("variant").isin([CONTROL_LABEL] + TREATMENT_LABELS))
)
if USE_EXPOSURE_FILTER:
    base = base.filter(F.col("is_exposed") == True)

# Одна строка на юзера
participants_all = (
    base.groupBy("participant_id", "participant_ext_id")
        .agg(
            F.min("event_date").alias("first_exposure_date"),
            F.first("variant").alias("ab_group"),
            F.first("bucket").alias("bucket_id"),
        )
        .select(
            F.col("participant_id").cast("long").alias("cookie_id"),
            F.col("participant_ext_id").cast("string").alias("device_id"),
            F.col("ab_group"),
            F.col("bucket_id").cast("long").alias("bucket_id"),
            F.col("first_exposure_date"),
        )
        # КРИТИЧНО: уникальный id бакета между группами
        .withColumn("bucket_uid",
                     F.concat_ws("_", F.col("ab_group"), F.col("bucket_id").cast("string")))
)

# Хеш-подвыборка по cookie_id (= participant_id — настоящая единица рандомизации)
participants_sampled = (
    participants_all
        .withColumn(
            "h",
            F.abs(F.xxhash64(F.concat(F.col("cookie_id").cast("string"),
                                       F.lit("|" + SAMPLE_SALT))))
        )
        .filter(F.col("h") % 100 < SAMPLE_PCT)
        .drop("h")
)
participants_sampled.cache()

n_total = participants_all.count()
n_sample = participants_sampled.count()
print(f"Participants: {n_total:,}, sampled ({SAMPLE_PCT}%): {n_sample:,}")

# Баланс групп + число уникальных бакетов (sanity: должно быть ≤ 256 на группу)
balance = (
    participants_sampled.groupBy("ab_group")
        .agg(
            F.countDistinct("cookie_id").alias("n_users"),
            F.countDistinct("bucket_uid").alias("n_buckets"),
        )
        .toPandas()
)
for r in balance.itertuples():
    print(f"  {r.ab_group:<20} users={r.n_users:,}, buckets={r.n_buckets}")

# Проверка: bucket_uid уникален между группами
total_buckets = participants_sampled.select("bucket_uid").distinct().count()
expected = int(balance["n_buckets"].sum())
print(f"Всего уникальных bucket_uid: {total_buckets} (ожидалось {expected})")
if total_buckets != expected:
    print("  ⚠️ bucket_uid не уникален между группами — баг!")

print(f"Время: {time.time()-t:.0f}s")

## Ячейка 5 — user_sessions → impressions с action-флагами

**Что происходит:** читаем сырые сессии из `user_sessions`, фильтруем по дате, выделяем релевантные показы (top-K по позиции внутри u2i-блока) и приджойниваем к каждому показу флаги действий:
- `was_clicked` — был ли клик по этому показу
- `was_btc` — был ли buyer_target_click (более ценное действие)
- `was_contact` — был ли запрос контакта продавца

**Особенности:**
- `TOP_K=100` — отсекаем длинный хвост показов далеко за экраном. Это стандартная практика в u2i-аналитике, потому что эффект изменений сильнее всего на верхних позициях
- Action-флаги джойнятся **на уровне (cookie_id, item_id, дата)** — это важно: один и тот же товар можно показать пользователю несколько раз, но клик по нему — событие более редкое
- `position` сохраняется как `int` для последующего использования в `BEN_weighted` (с DCG-весами)
- Колонки сохраняем минимальный набор: cookie_id, item_id, ab_group, position, was_*, microcat_id — чтобы pandas-материализация в Ячейке 6 не выходила за разумную память

**Производительность:** этот шаг почти всегда самый долгий в Spark-части (1–5 минут на прод-объёмах) из-за shuffle при join'ах. Дальнейшая работа после этого идёт уже в pandas.


In [ ]:

d_from = date.fromisoformat(DATE_FROM)
d_to   = date.fromisoformat(DATE_TO)
days = [(d_from + timedelta(i)).isoformat() for i in range((d_to - d_from).days + 1)]
print(f"Дней: {len(days)}")

impressions_raw_parts = []
for d in days:
    df_day = (
        spark.read.orc(f"{PATH_SESSIONS}/{d}")
        .select(F.col("device_id").cast("string").alias("device_id"), "u2i_serps")
    )
    # Двойной explode: u2i_serps → serp.items → items.*
    df_with_serp = df_day.withColumn("serp", F.explode("u2i_serps"))
    df_with_items = df_with_serp.select("device_id", F.col("serp.items").alias("items"))
    df_imp = df_with_items.withColumn("imp", F.explode("items"))
    
    # Action-флаги: из imp.ui.{click,btc,contact}. Это float (0.0 или 1.0+).
    imp = df_imp.select(
        "device_id",
        F.col("imp.id").cast("long").alias("item_id"),
        F.col("imp.pos").cast("int").alias("position"),
        F.col("imp.mcat_id").cast("long").alias("mcat_id"),
        (F.col("imp.ui.click")   > 0).cast("int").alias("was_clicked"),
        (F.col("imp.ui.btc")     > 0).cast("int").alias("was_btc"),
        (F.col("imp.ui.contact") > 0).cast("int").alias("was_contact"),
    )
    if TOP_K is not None:
        imp = imp.filter(F.col("position") <= TOP_K)
    impressions_raw_parts.append(imp)

impressions_raw = reduce(lambda a, b: a.unionByName(b), impressions_raw_parts)

if LOGCATS:
    impressions_filtered = (
        impressions_raw
            .join(F.broadcast(mcats_df),
                  impressions_raw.mcat_id == mcats_df.microcat_ext, "inner")
            .select("device_id", "item_id", "position",
                     "was_clicked", "was_btc", "was_contact")
    )
else:
    impressions_filtered = impressions_raw.select(
        "device_id", "item_id", "position",
        "was_clicked", "was_btc", "was_contact"
    )

## Ячейка 6 — Join с participants, дедуп, фильтр по логкатам, материализация

**Что происходит:**
1. **Inner join** impressions с participants — оставляем только показы пользователей которые в эксперименте
2. Если `LOGCATS` задан — **inner join с микрокатегориями**, оставляем только нужные логкаты
3. **Дедупликация** показов: `(cookie_id, item_id)` — оставляем только первый показ. Это нужно для корректности `unique_*` метрик: если один товар показан тебе 5 раз, для расчёта BEN/coverage важно лишь то что он был показан хотя бы раз
4. **Материализация в pandas** через `toPandas()` с Arrow

**Тонкость:** дедуп идёт перед материализацией, **после** join'ов. Это сильно уменьшает объём pandas-DataFrame и спасает driver от OOM на больших объёмах.

**Размер:** на 10% Fashion после всех фильтров — ~67M строк, ~3 GB pandas-памяти. На NUMA-bound железе чтение и преобразование занимает 2–3 минуты.


In [ ]:

t = time.time()
joined = (
    impressions_filtered.alias("i")
        .join(F.broadcast(participants_sampled.alias("p")), on="device_id", how="inner")
        .select(
            # cookie_id = participant_id (настоящая единица рандомизации/анализа)
            F.col("p.cookie_id").alias("cookie_id"),
            F.col("p.ab_group"),
            F.col("p.bucket_uid"),
            F.col("i.item_id"),
            F.col("i.position"),
            F.col("i.was_clicked"),
            F.col("i.was_btc"),
            F.col("i.was_contact"),
        )
)

# Дедуп: одна строка на (cookie, item), position=min, action-флаги=max
impressions = (
    joined.groupBy("cookie_id", "ab_group", "bucket_uid", "item_id")
          .agg(
              F.min("position").alias("position"),
              F.max("was_clicked").alias("was_clicked"),
              F.max("was_btc").alias("was_btc"),
              F.max("was_contact").alias("was_contact"),
          )
)
impressions.cache()
n_rows = impressions.count()

n_by_group = impressions.groupBy("ab_group").agg(
    F.countDistinct("cookie_id").alias("n_users"),
    F.countDistinct("item_id").alias("n_items"),
    F.count("*").alias("n_rows"),
    F.sum("was_clicked").alias("n_clicks"),
    F.sum("was_btc").alias("n_btc"),
    F.sum("was_contact").alias("n_contacts"),
).toPandas()

print(f"Строк: {n_rows:,}")
for r in n_by_group.itertuples():
    print(f"  {r.ab_group:<20} users={r.n_users:,}, items={r.n_items:,}, "
          f"rows={r.n_rows:,}")
    print(f"    clicks={r.n_clicks:,} ({100*r.n_clicks/max(r.n_rows,1):.2f}%), "
          f"btc={r.n_btc:,} ({100*r.n_btc/max(r.n_rows,1):.2f}%), "
          f"contacts={r.n_contacts:,} ({100*r.n_contacts/max(r.n_rows,1):.2f}%)")
print(f"Время: {time.time()-t:.0f}s")

# ПРОГНОЗ длительности (на основе реальных AA-замеров на Fashion 3% sample)
print("\n" + "=" * 60)
print("ПРОГНОЗ ДЛИТЕЛЬНОСТИ:")
print("=" * 60)
# AA-замер: 480s wall-time на iter @ ~14M строк pair_df × 150 perm
# = 480 / (14e6 × 150) ≈ 230 ns/row/perm на твоём железе (NUMA-bound)
# Production: после материализации pair_df имеет ~2/N_GROUPS от impressions
rows_per_pair = n_rows * 2 // 4  # control + одна treatment
ns_per_row_perm = 230e-9          # эмпирическая константа из AA-замера
honest_sec_per_pair = ns_per_row_perm * rows_per_pair * N_PERMUTATIONS_HONEST
bucket_sec_per_pair = 10
total_per_pair = honest_sec_per_pair + bucket_sec_per_pair
total_all = total_per_pair * len(TREATMENT_LABELS)
print(f"  На пару (~{rows_per_pair:,} строк × {N_PERMUTATIONS_HONEST} perm):")
print(f"    Unified perm:  ~{honest_sec_per_pair/60:.0f} мин (BEN + 4 unique + 4 unique_rel)")
print(f"    Bucket + IO:   ~{bucket_sec_per_pair/60:.1f} мин")
print(f"    ИТОГО:         ~{total_per_pair/60:.0f} мин")
print(f"\n  На все {len(TREATMENT_LABELS)} пар: ~{total_all/3600:.1f} часов (sequential)")
print(f"  При перезапуске пары пропускаются если уже посчитаны (резюм через HDFS-cache).")

if total_all > 6 * 3600:
    print("\n  ⚠️ ВНИМАНИЕ: прогноз > 6 часов.")
    print(f"     Рассмотри уменьшение:")
    print(f"     - N_PERMUTATIONS_HONEST = 200 (даст ускорение 2.5×)")
    print(f"     - SAMPLE_PCT = {max(1, SAMPLE_PCT//2)} (даст ускорение 2×)")
    print(f"     При падении → Run All подхватит с того же места из чекпоинтов.")

## Ячейка 7 — Unified permutation-тест (главное вычислительное ядро)

**Что происходит:** функция `honest_user_permutation_test` — основной численный метод. Считает 10 метрик (BEN, BEN_weighted, 4 unique абсолютных, 4 unique относительных) **за один permutation loop**.

### Метод

Для каждой пары `control vs treatment`:

1. **delta_obs = T - C** (наблюдаемая разница для каждой метрики)
2. Для `b = 1..N_perm`:
   - Случайно перемешиваем метки группы на уровне `cookie_id`
   - Пересчитываем все 10 метрик при новых метках
   - Получаем `delta_b`
3. **p-value** = доля `|delta_b - mean(delta_b)| ≥ |delta_obs - mean(delta_b)|` (centered p-value, устойчиво к смещениям)
4. **H₀-границы** = квантили [α/2, 1−α/2] permutation-распределения

### Почему так

- **Permutation, не bootstrap.** Для unique-метрик (count-distinct) bootstrap даёт смещённый CI из-за нелинейности — он не пересчитывает `q(item)` корректно при resample with replacement. Permutation работает корректно для всех 10 метрик одним методом.
- **На уровне юзеров, не показов.** Buyer-level — это то что просит [определение метрики BEN](https://en.wikipedia.org/wiki/Permutation_test). Permutation на показах нарушает зависимость BEN от других пользователей через `q(item)`.
- **Один loop вместо 10.** В каждой permutation-итерации все 10 метрик считаются параллельно, потому что `q(item)` (доля видевших) пересчитывается один раз и переиспользуется для всех метрик.

### Оптимизации в numpy

- `unique_items` через `(c_A > 0).sum()` — **бесплатно** (c_A уже считается для BEN)
- Action-subsets (`clicks`, `btc`, `contacts`) выносятся за loop — pre-extract один раз
- `1/k_u`, `1/sum_w_user` — pre-compute
- `nlq_per_row` через `nlq_B[i_arr]` + in-place правка для группы A — экономит 400+ MB на 25M строк
- HDFS-кэш материализации pair_df через `cache_path` — при повторном запуске после падения не считаем pair_df заново

### Дополнительно

Внутри функции каждые `ping_every` итераций — keep-alive ping в Spark. На прод-объёмах одна перестановка ≈ 5 секунд (NUMA-bound), всего 500 perm × 3 пары ≈ 2 часа реальных.


In [ ]:
#
# В одном permutation loop одновременно считаются 10 метрик:
#   - BEN (q_per_group)              — основная diversity-метрика
#   - BEN_weighted                    — взвешенная по позиции
#   - unique_items                    — # уникальных items в группе
#   - unique_clicked_items            — # уникальных кликнутых items
#   - unique_btc_items                — # уникальных items в избранном
#   - unique_contacted_items          — # уникальных items с контактами
#   - unique_items_rel                — относительное изменение (T-C)/C  (NEW v11)
#   - unique_clicked_items_rel                                            (NEW v11)
#   - unique_btc_items_rel                                                (NEW v11)
#   - unique_contacted_items_rel                                          (NEW v11)
#
# Для каждой метрики:
#   delta_obs = T - C  (для *_rel: (T - C) / C)
#   permutation distribution = распределение разности под H0
#   p_value (centered) = доля |perm - mean(perm)| >= |delta_obs - mean(perm)|
#
# Единый loop экономит время: одна перестановка → 10 метрик за один проход.
#
# === Оптимизации v11 относительно v10 (gold-test пройден на синтетике) ===
# 1. unique_items вычисляется через (c_A>0).sum() — БЕСПЛАТНО,
#    т.к. c_A/c_B уже посчитаны для BEN.
# 2. action-subsets (clicks/btc/contacts) предвычисляются один раз
#    вне permutation loop. Cущественное ускорение для разреженных метрик.
# 3. 1/k_u и 1/sum_w_user — pre-compute раз вместо повторных делений.
# 4. nlq_per_row аллоцируется один раз через `nlq_B[i_arr]` + in-place
#    правка, что экономит ~400MB пиковой памяти на 25M строк.
#
# === Замечания по *_rel (NEW) ===
# 1. Под H0 относительная метрика монотонно связана с абсолютной
#    (т.к. nC_perm стабильна между permutation), поэтому AA-валидация
#    показывает корреляцию p-value abs↔rel ≈ 0.998.
#    *_rel даёт то же статистическое решение, но в более интерпретируемой
#    шкале (% от уровня control).
# 2. Если в перестановке nC_perm = 0 (например, при пустой control-группе
#    по action), элемент distribution = NaN и игнорируется при расчёте
#    p-value и квантилей.

def honest_user_permutation_test(pair_df_sp, control_label, treatment_label,
                                   n_perm=500, seed=42, ping_every=50, spark_ctx=None,
                                   compute_ci=True, ci_metrics="all", alpha=0.05,
                                   cache_path=None):
    """Unified permutation test для BEN, BEN_weighted, 4 unique и 4 unique_rel.
    
    compute_ci: если True, для метрик из ci_metrics в результат добавляются
        h0_lower / h0_upper (квантили distribution под H0) и
        is_significant (флаг "delta_obs выходит за квантили").
    ci_metrics: 
        - "all": для всех 10 метрик
        - None: ни для какой
        - список: только для указанных
    alpha: уровень значимости для квантилей под H0 (по умолчанию 0.05).
    
    cache_path: HDFS-путь для кэша материализации pair_df_sp.
        Если задан и существует — читаем оттуда вместо toPandas().
        Если задан и не существует — пишем туда после материализации.
        Если None — без кэша (как раньше).
    
    Keep-alive: каждые `ping_every` итераций — лёгкий Spark-запрос.
    """
    t_m = time.time()
    
    # === Кэш материализации pair_df ===
    if cache_path is not None and _hdfs_path_exists(cache_path):
        print(f"      Cache HIT: {cache_path}", flush=True)
        imp = spark.read.parquet(cache_path).toPandas()
        print(f"      Прочитано из кэша: {time.time()-t_m:.0f}s, {len(imp):,} строк")
    else:
        if cache_path is not None:
            print(f"      Cache MISS, пишем в {cache_path}", flush=True)
            pair_df_sp.select(
                "cookie_id", "item_id", "ab_group", "position",
                "was_clicked", "was_btc", "was_contact",
            ).write.mode("overwrite").parquet(cache_path)
            print(f"      Кэш записан за {time.time()-t_m:.0f}s, читаем обратно...", flush=True)
            t2 = time.time()
            imp = spark.read.parquet(cache_path).toPandas()
            print(f"      Прочитан кэш: {time.time()-t2:.0f}s, {len(imp):,} строк")
        else:
            imp = pair_df_sp.select(
                "cookie_id", "item_id", "ab_group", "position",
                "was_clicked", "was_btc", "was_contact",
            ).toPandas()
            print(f"      Материализация: {time.time()-t_m:.0f}s, {len(imp):,} строк")
    
    from pandas import factorize as _fact
    u_arr = _fact(imp["cookie_id"], sort=False)[0].astype(np.int32)
    i_arr = _fact(imp["item_id"], sort=False)[0].astype(np.int32)
    pos_arr = imp["position"].values.astype(np.float64)
    clicks_arr   = imp["was_clicked"].values.astype(np.int8)
    btc_arr      = imp["was_btc"].values.astype(np.int8)
    contacts_arr = imp["was_contact"].values.astype(np.int8)
    n_users = int(u_arr.max()) + 1
    n_items = int(i_arr.max()) + 1
    
    g_codes, g_uniq = _fact(imp["ab_group"], sort=False)
    g_codes = g_codes.astype(np.int8)
    user_group = np.empty(n_users, dtype=np.int8)
    order = np.argsort(u_arr, kind="stable")
    u_sorted = u_arr[order]
    first = np.concatenate([[0], np.where(np.diff(u_sorted) != 0)[0] + 1])
    user_group[u_sorted[first]] = g_codes[order][first]
    c_idx = np.where(g_uniq == control_label)[0][0]
    user_group_norm = np.where(user_group == c_idx, 0, 1).astype(np.int8)
    del imp, order, u_sorted, first
    
    # ============================================================
    # Pre-compute (один раз)
    # ============================================================
    c_total = np.bincount(i_arr, minlength=n_items).astype(np.float64)
    k_u = np.bincount(u_arr, minlength=n_users).astype(np.float64)
    w_arr = 1.0 / np.log2(pos_arr + 1)
    sum_w_user = np.bincount(u_arr, weights=w_arr, minlength=n_users)
    eps = 1e-8
    inv_k_u = 1.0 / np.maximum(k_u, 1.0)
    inv_sum_w = 1.0 / np.maximum(sum_w_user, 1e-12)
    
    # Pre-extract action-subsets (sparse → большая экономия)
    cl_mask_b = clicks_arr == 1
    bt_mask_b = btc_arr == 1
    co_mask_b = contacts_arr == 1
    i_cl = i_arr[cl_mask_b]; u_cl = u_arr[cl_mask_b]
    i_bt = i_arr[bt_mask_b]; u_bt = u_arr[bt_mask_b]
    i_co = i_arr[co_mask_b]; u_co = u_arr[co_mask_b]
    print(f"      Pre-extract: clicks={len(i_cl):,}, btc={len(i_bt):,}, contacts={len(i_co):,}")
    
    abs_metric_names = [
        "ben", "ben_weighted",
        "unique_items", "unique_clicked_items", "unique_btc_items", "unique_contacted_items",
    ]
    rel_metric_names = [
        "unique_items_rel", "unique_clicked_items_rel",
        "unique_btc_items_rel", "unique_contacted_items_rel",
    ]
    metric_names = abs_metric_names + rel_metric_names
    
    def _count_uniq_subset(i_subset, u_subset, group_vec, side):
        """# уникальных items в группе `side` (0 или 1) внутри subset."""
        if len(i_subset) == 0:
            return 0
        rg = group_vec[u_subset]
        sel = (rg == side)
        if not sel.any():
            return 0
        c = np.bincount(i_subset[sel], minlength=n_items)
        return int((c > 0).sum())
    
    def compute_all_metrics(group_vec):
        """Возвращает 10 значений: 6 абсолютных + 4 относительных."""
        row_group = group_vec[u_arr]
        mask_A = row_group == 0  # control
        n_A = int((group_vec == 0).sum())
        n_B = n_users - n_A
        inv_nA = 1.0 / max(n_A, 1)
        inv_nB = 1.0 / max(n_B, 1)
        
        # --- BEN / BEN_weighted (с in-place оптимизацией памяти) ----
        c_A = np.bincount(i_arr[mask_A], minlength=n_items).astype(np.float64)
        c_B = c_total - c_A
        nlq_A = -np.log(np.maximum(c_A * inv_nA, eps))
        nlq_B = -np.log(np.maximum(c_B * inv_nB, eps))
        # nlq_per_row: создаём как gather из B, потом правим строки A.
        # Один peak-allocation вместо трёх в np.where(mask_A, ...).
        nlq_per_row = nlq_B[i_arr]
        nlq_per_row[mask_A] = nlq_A[i_arr[mask_A]]
        sum_nlq = np.bincount(u_arr, weights=nlq_per_row, minlength=n_users)
        np.multiply(nlq_per_row, w_arr, out=nlq_per_row)  # переиспользуем буфер
        sum_wnlq = np.bincount(u_arr, weights=nlq_per_row, minlength=n_users)
        del nlq_per_row, nlq_A, nlq_B
        ben_u = sum_nlq * inv_k_u
        ben_w_u = sum_wnlq * inv_sum_w
        
        mA_u = group_vec == 0
        d_ben  = float(ben_u[~mA_u].mean()  - ben_u[mA_u].mean())
        d_benw = float(ben_w_u[~mA_u].mean() - ben_w_u[mA_u].mean())
        
        # --- unique_items: бесплатно из c_A, c_B ----
        nA_uni = int((c_A > 0).sum())
        nB_uni = int((c_B > 0).sum())
        d_uni = nB_uni - nA_uni
        
        # --- unique_*_items: на разреженных subset ----
        nA_cl = _count_uniq_subset(i_cl, u_cl, group_vec, 0)
        nB_cl = _count_uniq_subset(i_cl, u_cl, group_vec, 1)
        d_clk = nB_cl - nA_cl
        
        nA_bt = _count_uniq_subset(i_bt, u_bt, group_vec, 0)
        nB_bt = _count_uniq_subset(i_bt, u_bt, group_vec, 1)
        d_btc = nB_bt - nA_bt
        
        nA_co = _count_uniq_subset(i_co, u_co, group_vec, 0)
        nB_co = _count_uniq_subset(i_co, u_co, group_vec, 1)
        d_con = nB_co - nA_co
        
        # --- relative-метрики: (T - C) / C ----
        d_uni_rel = (nB_uni - nA_uni) / nA_uni if nA_uni > 0 else np.nan
        d_clk_rel = (nB_cl - nA_cl) / nA_cl if nA_cl > 0 else np.nan
        d_btc_rel = (nB_bt - nA_bt) / nA_bt if nA_bt > 0 else np.nan
        d_con_rel = (nB_co - nA_co) / nA_co if nA_co > 0 else np.nan
        
        return (d_ben, d_benw, d_uni, d_clk, d_btc, d_con,
                d_uni_rel, d_clk_rel, d_btc_rel, d_con_rel)
    
    # Observed deltas
    obs = compute_all_metrics(user_group_norm)
    d_obs = dict(zip(metric_names, obs))
    
    # === Точечные оценки nunique_C и nunique_T для unique-метрик ===
    # Нужны для расчёта относительной метрики (T - C) / C для отчёта.
    # Считаются один раз на observed данных, не замедляют permutation loop.
    unique_levels = {}
    for name, i_sub, u_sub in [
        ("unique_items", i_arr, u_arr),
        ("unique_clicked_items", i_cl, u_cl),
        ("unique_btc_items", i_bt, u_bt),
        ("unique_contacted_items", i_co, u_co),
    ]:
        nC = _count_uniq_subset(i_sub, u_sub, user_group_norm, 0)
        nT = _count_uniq_subset(i_sub, u_sub, user_group_norm, 1)
        unique_levels[name] = (nC, nT)
    
    # Permutation loop
    rng = np.random.default_rng(seed)
    deltas = {name: np.empty(n_perm) for name in metric_names}
    group_p = user_group_norm.copy()
    t0 = time.time()
    for b in range(n_perm):
        rng.shuffle(group_p)
        ds = compute_all_metrics(group_p)
        for j, name in enumerate(metric_names):
            deltas[name][b] = ds[j]
        
        if (b + 1) % ping_every == 0:
            el = time.time() - t0
            eta = el / (b + 1) * (n_perm - b - 1)
            print(f"      perm {b+1}/{n_perm}  {el/(b+1)*1000:.0f} ms/it, "
                  f"ETA {eta:.0f}s", flush=True)
            if spark_ctx is not None:
                try:
                    _ = spark_ctx.parallelize([1, 2, 3]).count()
                    print(f"        [spark ping OK]", flush=True)
                except Exception as e:
                    print(f"        [spark ping FAILED: {type(e).__name__}]", flush=True)
    
    # === Resolve ci_metrics ===
    if not compute_ci or ci_metrics is None:
        ci_set = set()
    elif ci_metrics == "all":
        ci_set = set(metric_names)
    elif isinstance(ci_metrics, (list, tuple)):
        ci_set = set(ci_metrics)
        unknown = ci_set - set(metric_names)
        if unknown:
            print(f"      ⚠ ci_metrics: неизвестные метрики {unknown} — игнорируются")
            ci_set = ci_set - unknown
    else:
        ci_set = set()
    
    # p-value (centered) и distribution under H0
    results = {}
    for name in metric_names:
        d = float(d_obs[name])
        arr = deltas[name]
        # Для *_rel может быть nan если в перестановке nA=0. Игнорируем.
        arr_clean = arr[~np.isnan(arr)]
        n_clean = len(arr_clean)
        mean_p = float(np.mean(arr_clean)) if n_clean > 0 else 0.0
        std_p  = float(np.std(arr_clean, ddof=1)) if n_clean > 1 else 0.0
        
        # centered p-value
        if n_clean == 0 or np.isnan(d):
            p = 1.0
        elif d != 0:
            p = (np.sum(np.abs(arr_clean - mean_p) >= abs(d - mean_p)) + 1) / (n_clean + 1)
        else:
            p = 1.0
        
        result_entry = {
            "delta_obs":      d,
            "p_value":        float(min(p, 1.0)),
            "perm_mean":      mean_p,
            "perm_std":       std_p,
            "n_permutations": n_perm,
            "n_valid_perm":   n_clean,
        }
        
        # Уровни и относит. отчёт — для unique-метрик (включая *_rel)
        base = name.replace("_rel", "")
        if base in unique_levels:
            nC, nT = unique_levels[base]
            result_entry["control_level"]    = int(nC)
            result_entry["treatment_level"]  = int(nT)
            if nC > 0:
                rel = (nT - nC) / nC
                result_entry["relative_change"]     = float(rel)
                result_entry["relative_change_pct"] = float(rel * 100)
            else:
                result_entry["relative_change"]     = None
                result_entry["relative_change_pct"] = None
        
        if name in ci_set:
            if n_clean > 0:
                h0_lo = float(np.percentile(arr_clean, alpha/2 * 100))
                h0_hi = float(np.percentile(arr_clean, (1 - alpha/2) * 100))
                result_entry["h0_lower"]       = h0_lo
                result_entry["h0_upper"]       = h0_hi
                result_entry["is_significant"] = bool(
                    (not np.isnan(d)) and ((d < h0_lo) or (d > h0_hi))
                )
                # Относительные H0-границы для абсолютных unique-метрик: H0_bound / nC
                if base in unique_levels and not name.endswith("_rel"):
                    nC, _ = unique_levels[base]
                    if nC > 0:
                        result_entry["h0_lower_pct"] = float(h0_lo / nC * 100)
                        result_entry["h0_upper_pct"] = float(h0_hi / nC * 100)
            else:
                result_entry["h0_lower"] = np.nan
                result_entry["h0_upper"] = np.nan
                result_entry["is_significant"] = False
        
        results[name] = result_entry
    
    return results

## Ячейка 8 — analyze_pair: оркестратор анализа одной пары

**Что происходит:** `analyze_pair(impressions_all, control, treatment, n_perm, seed, cache_root)` делает все шаги для одной пары:

1. **Фильтр и кэширование pair_df** — оставляем только записи `control` и `treatment`. `pair_df.cache()` материализует Spark-DataFrame в memory executors.
2. **Считаем размер пары** — пользователи / items / показы / actions, печатаем для sanity-check.
3. **Вызов `honest_user_permutation_test`** — основное permutation-вычисление. Возвращает dict с результатами для всех 10 метрик.
4. **Разделение результатов** на 3 группы:
   - `ben_permutation`: BEN, BEN_weighted (главная пара)
   - `unique_permutation`: 4 unique абсолютных
   - `unique_rel_permutation`: 4 unique относительных
5. **Unpersist** pair_df чтобы освободить память executors.

**Особенность:** функция возвращает «сырые» результаты с `_distrib` (массивами permutation-распределений). Это нужно для последующей работы с CI/p-value в `Ячейке 10`. При сохранении в JSON массивы выкидываются.


In [ ]:

def analyze_pair(impressions_all, control_label, treatment_label,
                  n_perm_honest, seed, cache_root=None):
    """Анализ одной пары control vs treatment.
    
    cache_root: HDFS-директория для кэшей материализации pair_df.
        Если задана — внутри неё создаётся подпапка для пары.
    
    Метод: Unified honest user-level permutation —
        BEN, BEN_weighted, и 4 unique-метрики (+ 4 *_rel) одним loop'ом.
    """
    print("\n" + "=" * 70)
    print(f"  Пара: {control_label}  vs  {treatment_label}")
    print("=" * 70)
    
    pair_df = impressions_all.filter(F.col("ab_group").isin([control_label, treatment_label]))
    pair_df.cache()
    stats_pair = pair_df.groupBy("ab_group").agg(
        F.countDistinct("cookie_id").alias("n_users"),
        F.countDistinct("item_id").alias("n_items"),
        F.count("*").alias("n_rows"),
        F.sum("was_clicked").alias("n_clicks"),
        F.sum("was_btc").alias("n_btc"),
        F.sum("was_contact").alias("n_contacts"),
    ).toPandas()
    print("\n  Объём:")
    for r in stats_pair.itertuples():
        print(f"    {r.ab_group:<18} users={r.n_users:,}, items={r.n_items:,}, "
              f"rows={r.n_rows:,}")
        print(f"      clicks={r.n_clicks:,}, btc={r.n_btc:,}, contacts={r.n_contacts:,}")
    
    # Кэш материализации pair_df
    pair_cache_path = None
    if cache_root is not None:
        # Безопасный для путей label
        safe_t = treatment_label.replace("/", "_").replace(" ", "_")
        pair_cache_path = f"{cache_root}/pair_{safe_t}.parquet"
    
    # === UNIFIED HONEST USER PERM: BEN + BEN_weighted + 4 unique + 4 unique_rel ===
    print(f"\n  Unified honest user-level perm ({n_perm_honest} permutations):")
    print(f"        Метрики: BEN, BEN_weighted, unique_items/clicked/btc/contacts")
    t_h = time.time()
    results_unified = honest_user_permutation_test(
        pair_df, control_label, treatment_label,
        n_perm=n_perm_honest, seed=seed,
        ping_every=PING_EVERY_ITERS, spark_ctx=spark.sparkContext,
        compute_ci=COMPUTE_CI, ci_metrics=CI_METRICS, alpha=ALPHA,
        cache_path=pair_cache_path,
    )
    print(f"  Done за {time.time()-t_h:.0f}s")
    
    # Разделяем для удобства принта
    ben_results    = {k: results_unified[k] for k in ["ben", "ben_weighted"]}
    unique_results = {k: results_unified[k] for k in
                       ["unique_items", "unique_clicked_items",
                        "unique_btc_items", "unique_contacted_items"]}
    unique_rel_results = {k: results_unified[k] for k in
                       ["unique_items_rel", "unique_clicked_items_rel",
                        "unique_btc_items_rel", "unique_contacted_items_rel"]}
    
    print(f"\n  BEN-метрики:")
    for col, r in ben_results.items():
        if "h0_lower" in r:
            sig_flag = "*" if r["is_significant"] else " "
            print(f"    {col:<15} Δ={r['delta_obs']:>+12.6f}  p={r['p_value']:.4f}  "
                  f"H0=[{r['h0_lower']:+.6f},{r['h0_upper']:+.6f}] {sig_flag}")
        else:
            print(f"    {col:<15} Δ={r['delta_obs']:>+12.6f}  p={r['p_value']:.4f}")
    
    print(f"\n  Unique-метрики (абсолютные):")
    for col, r in unique_results.items():
        rel_str = ""
        if r.get("relative_change_pct") is not None:
            rel_str = f"  ({r['relative_change_pct']:+.2f}%)"
        if "h0_lower" in r:
            sig_flag = "*" if r["is_significant"] else " "
            h0_pct_str = ""
            if "h0_lower_pct" in r:
                h0_pct_str = f"  H0%=[{r['h0_lower_pct']:+.2f}%,{r['h0_upper_pct']:+.2f}%]"
            print(f"    {col:<25} Δ={r['delta_obs']:>+12,.0f}{rel_str}  p={r['p_value']:.4f}  "
                  f"H0=[{r['h0_lower']:+,.0f},{r['h0_upper']:+,.0f}]{h0_pct_str} {sig_flag}")
        else:
            print(f"    {col:<25} Δ={r['delta_obs']:>+12,.0f}{rel_str}  p={r['p_value']:.4f}")
    
    print(f"\n  Unique-метрики (относительные, NEW v11):")
    for col, r in unique_rel_results.items():
        if "h0_lower" in r and not np.isnan(r["h0_lower"]):
            sig_flag = "*" if r["is_significant"] else " "
            print(f"    {col:<32} Δ={r['delta_obs']*100:>+9.3f}%  p={r['p_value']:.4f}  "
                  f"H0=[{r['h0_lower']*100:+.3f}%,{r['h0_upper']*100:+.3f}%] {sig_flag}")
        else:
            print(f"    {col:<32} Δ={r['delta_obs']*100:>+9.3f}%  p={r['p_value']:.4f}")
    
    pair_df.unpersist()
    return {
        "control": control_label,
        "treatment": treatment_label,
        "ben_permutation":         ben_results,           # BEN, BEN_weighted
        "unique_permutation":      unique_results,        # 4 unique абс.
        "unique_rel_permutation":  unique_rel_results,    # 4 unique отн.
    }

## Ячейка 9 — Главный цикл: прогон всех пар с resume и resilience

**Что происходит:** проходим по всем `TREATMENT_LABELS` и считаем `analyze_pair` для каждой. Перед каждой парой проверяем что Spark жив (`ensure_spark_alive`), при падении пересоздаём session.

### Resume через HDFS-чекпоинты

1. Перед main loop проверяем какие пары **уже посчитаны** — для каждой `treatment` смотрим существует ли `pair_{treatment}.json` в `PATH_RESULTS`.
2. Уже посчитанные пары загружаем из чекпоинта, не пересчитываем.
3. Считаем только недостающие.

После каждой завершённой пары:
- Сохраняем JSON с p-value, delta, H0-границами для каждой метрики
- Сохраняем pair_df.parquet в `CACHE_ROOT` (это делается внутри `honest_user_permutation_test`)

**Что делать если упало посреди пары:**
- Просто `Run All` — Ячейка 1 пересоздаёт Spark, Ячейка 6 читает кэш материализации, Ячейка 9 пропускает уже завершённые пары
- Текущая (упавшая) пара пересчитается с нуля, но материализация pair_df возьмётся из кэша

**Что делать если хочешь форс-пересчитать:**
- `FORCE_REBUILD_ALL = True` в этой же ячейке
- Или вручную удали JSON-файлы из `PATH_RESULTS` и `pair_*.parquet` из `CACHE_ROOT`


In [ ]:
#
# Фичи:
#   1. Кэш материализации pair_df на HDFS — при перезапуске не считаем
#      pair_df снова, читаем готовый parquet.
#   2. Чекпоинт результата каждой пары — при перезапуске пропускаем
#      уже посчитанные пары.
#   3. Resilient session — между парами проверяем что Spark жива,
#      пересоздаём при падении.
#
# ВАЖНО про падения посреди пары:
#   Если Spark упадёт ВНУТРИ analyze_pair (например при материализации),
#   лучшая стратегия — Kernel → Restart → Run All. Кэш и checkpoint
#   автоматически подхватят прогресс: завершённые пары пропустятся,
#   материализация текущей пары — из cache_path если успела записаться.
#
# Чтобы форсированно пересчитать конкретную пару — удали соответствующий
# pair_*.json в PATH_RESULTS и pair_*.parquet в CACHE_ROOT.
# Чтобы всё пересчитать — поставь FORCE_REBUILD_ALL = True.
FORCE_REBUILD_ALL = False

print("=" * 60)
print(f"Анализ {len(TREATMENT_LABELS)} пар")
print("=" * 60)

# Кэш-корень для материализаций (внутри HDFS)
CACHE_ROOT = f"{HDFS_HOME}/ben_cache/exp{EXPERIMENT_ID}/topk_{TOP_K}_{_mode}_lc-{_logcats_tag}_v11"
print(f"Cache root: {CACHE_ROOT}")
print(f"Results path: {PATH_RESULTS}")

def _load_pair_result(treatment):
    """Попытаться загрузить уже сохранённый результат пары из чекпоинта."""
    path = f"{PATH_RESULTS}/pair_{treatment}.json"
    if not _hdfs_path_exists(path):
        return None
    try:
        rows = spark.read.text(path).collect()
        payload = "\n".join(r["value"] for r in rows)
        return json.loads(payload)
    except Exception as e:
        print(f"  [load_pair {treatment}] {e}")
        return None


# Восстанавливаем посчитанные пары
all_results = {}
t_all = time.time()

if not FORCE_REBUILD_ALL:
    for treatment in TREATMENT_LABELS:
        cached = _load_pair_result(treatment)
        if cached is not None:
            print(f"\n  Пара {treatment}: HIT (из чекпоинта)")
            all_results[treatment] = cached

# Считаем недостающие пары
remaining = [t for t in TREATMENT_LABELS if t not in all_results]
print(f"\nПосчитано из чекпоинта: {len(all_results)}/{len(TREATMENT_LABELS)}")
print(f"Осталось: {len(remaining)} пар")

for treatment in remaining:
    t_pair = time.time()
    
    # Перед каждой парой — проверяем session, пересоздаём если умерла
    spark, recreated = ensure_spark_alive()
    if recreated:
        print(f"  [resilient] Пересоздана session перед парой {treatment}")
    
    res = analyze_pair(
        impressions, CONTROL_LABEL, treatment,
        n_perm_honest=N_PERMUTATIONS_HONEST,
        seed=SEED,
        cache_root=CACHE_ROOT,
    )
    all_results[treatment] = res
    print(f"\n  Пара {treatment}: {time.time()-t_pair:.0f}s")
    
    # Чекпоинт после каждой пары — повторные попытки если session упала
    pair_summary = {
        "control": res["control"],
        "treatment": res["treatment"],
        "ben_permutation": {
            k: {kk: vv for kk, vv in v.items() if kk != "_distrib"}
            for k, v in res["ben_permutation"].items()
        },
        "unique_permutation": {
            k: {kk: vv for kk, vv in v.items() if kk != "_distrib"}
            for k, v in res["unique_permutation"].items()
        },
        "unique_rel_permutation": {
            k: {kk: vv for kk, vv in v.items() if kk != "_distrib"}
            for k, v in res["unique_rel_permutation"].items()
        },
    }
    pair_path = f"{PATH_RESULTS}/pair_{treatment}.json"
    saved = False
    for attempt in range(3):
        try:
            spark.createDataFrame([(json.dumps(pair_summary, indent=2,
                                                ensure_ascii=False, default=str),)],
                                    ["summary"]).coalesce(1) \
                .write.mode("overwrite").text(pair_path)
            print(f"  Checkpoint: {pair_path}")
            saved = True
            break
        except Exception as e:
            print(f"  Checkpoint err (attempt {attempt+1}/3): {e}")
            spark, _ = ensure_spark_alive()
    if not saved:
        print(f"  [WARN] Не удалось записать чекпоинт {pair_path} — результат в памяти есть.")

print(f"\nВсе пары готовы за {time.time()-t_all:.0f}s")

## Ячейка 10 — Сводный отчёт по всем парам

**Что происходит:** проходим по всем результатам из `all_results`, печатаем читаемую сводку для каждой пары:
- BEN-метрики: BEN, BEN_weighted с delta, p-value, H₀-границами
- Unique-метрики (абсолютные): unique_items / clicked / btc / contacted с delta, относительным изменением, p-value, H₀-границами
- Unique-метрики (относительные NEW v11): те же 4, но в относительной шкале (T-C)/C

**Звёздочки `*`** — флаг `is_significant` (delta_obs выходит за квантили H₀ на уровне α=0.05).

**Особенность форматирования:**
- Абсолютные unique подаются как целые числа (количество товаров) с их относительным изменением в скобках
- Относительные unique подаются в процентах (`+6.61%`)
- H₀-границы тоже в процентах для удобства интерпретации

Это последняя «человеческая» секция — после неё идёт сохранение в JSON.


In [ ]:

print("\n" + "=" * 70)
print("ИТОГИ v10 (unified permutation, без bootstrap)")
print("=" * 70)

# Multiple testing correction
if MULTIPLE_CORRECTION == "bonferroni":
    alpha_corrected = ALPHA / len(TREATMENT_LABELS)
    print(f"Bonferroni: α = {ALPHA} / {len(TREATMENT_LABELS)} = {alpha_corrected:.4f}")
else:
    alpha_corrected = ALPHA
    print(f"No correction, α = {ALPHA}")

for treatment, res in all_results.items():
    print(f"\n{'=' * 70}")
    print(f"{CONTROL_LABEL} vs {treatment}:")
    print(f"{'=' * 70}")
    
    print(f"\n  [BEN-метрики, honest user perm + q_per_group] MAIN")
    for col, r in res["ben_permutation"].items():
        sig = "*" if r["p_value"] < alpha_corrected else " "
        if "h0_lower" in r:
            print(f"    {col:<25} Δ={r['delta_obs']:>+12.6f}  p={r['p_value']:.4f}  "
                  f"H0=[{r['h0_lower']:+.4f},{r['h0_upper']:+.4f}] {sig}")
        else:
            print(f"    {col:<25} Δ={r['delta_obs']:>+12.6f}  p={r['p_value']:.4f}  {sig}")
    
    print(f"\n  [Unique-метрики, honest user perm] MAIN")
    for col, r in res["unique_permutation"].items():
        sig = "*" if r["p_value"] < alpha_corrected else " "
        rel_str = ""
        if r.get("relative_change_pct") is not None:
            rel_str = f"  ({r['relative_change_pct']:+.2f}%)"
        if "h0_lower" in r:
            h0_pct_str = ""
            if "h0_lower_pct" in r:
                h0_pct_str = f"  H0%=[{r['h0_lower_pct']:+.2f}%,{r['h0_upper_pct']:+.2f}%]"
            print(f"    {col:<25} Δ={r['delta_obs']:>+12,.0f}{rel_str}  p={r['p_value']:.4f}  "
                  f"H0=[{r['h0_lower']:+,.0f},{r['h0_upper']:+,.0f}]{h0_pct_str} {sig}")
        else:
            print(f"    {col:<25} Δ={r['delta_obs']:>+12,.0f}{rel_str}  p={r['p_value']:.4f}  {sig}")
    
    print(f"\n  [Unique-метрики ОТНОСИТЕЛЬНЫЕ, NEW v11] (T-C)/C")
    for col, r in res["unique_rel_permutation"].items():
        sig = "*" if r["p_value"] < alpha_corrected else " "
        if "h0_lower" in r and not (isinstance(r["h0_lower"], float) and np.isnan(r["h0_lower"])):
            print(f"    {col:<32} Δ={r['delta_obs']*100:>+9.3f}%  p={r['p_value']:.4f}  "
                  f"H0=[{r['h0_lower']*100:+.3f}%,{r['h0_upper']*100:+.3f}%]  {sig}")
        else:
            print(f"    {col:<32} Δ={r['delta_obs']*100:>+9.3f}%  p={r['p_value']:.4f}  {sig}")

## Ячейка 11 — Сохранение результатов в HDFS + cleanup

**Что происходит:**
1. Собираем `all_summary` — JSON-структуру с метаданными (даты, параметры, версия) и результатами всех пар (без массивов `_distrib`)
2. Записываем в HDFS как одиночный текстовый файл `all_summary.json` в `PATH_RESULTS`
3. **Cleanup**: `pair_df.unpersist()` для всех cache'ов

**Зачем JSON:**
- Удобно для последующей загрузки в pandas через `spark.read.text()` или просто `hdfs cat`
- Все p-value, delta, H₀-границы в структурированном виде — можно строить любые сравнения post-hoc
- Метаданные сохранены — через год по `version` и параметрам легко понять что это был за прогон

**Структура:**
```json
{
  "experiment_id": 30895,
  "date_from": "2026-03-13", "date_to": "2026-03-18",
  "logcats": ["Goods.Fashion"],
  "sample_pct": 10,
  "n_perm_honest": 500,
  "alpha": 0.05,
  "version": "v11_unified_permutation_with_unique_rel_metrics",
  "results": {
    "v1i_personal": {
      "control": "control", "treatment": "v1i_personal",
      "ben_permutation": {...},
      "unique_permutation": {...},
      "unique_rel_permutation": {...}
    },
    ...
  }
}
```

После этой ячейки можно идти в любой ноутбук и читать готовые результаты для построения графиков, сравнений между прогонами и так далее.


In [ ]:

try:
    all_summary = {
        "experiment_id": EXPERIMENT_ID,
        "date_from": DATE_FROM, "date_to": DATE_TO,
        "logcats": LOGCATS,
        "sample_pct": SAMPLE_PCT,
        "top_k": TOP_K,
        "use_exposure_filter": USE_EXPOSURE_FILTER,
        "n_perm_honest": N_PERMUTATIONS_HONEST,
        "alpha": ALPHA,
        "multiple_correction": MULTIPLE_CORRECTION,
        "alpha_corrected": alpha_corrected,
        "version": "v11_unified_permutation_with_unique_rel_metrics",
        "results": {
            t: {
                "control": r["control"], "treatment": r["treatment"],
                "ben_permutation": {
                    k: {kk: vv for kk, vv in v.items()}
                    for k, v in r["ben_permutation"].items()
                },
                "unique_permutation": {
                    k: {kk: vv for kk, vv in v.items()}
                    for k, v in r["unique_permutation"].items()
                },
                "unique_rel_permutation": {
                    k: {kk: vv for kk, vv in v.items()}
                    for k, v in r["unique_rel_permutation"].items()
                },
            }
            for t, r in all_results.items()
        },
    }
    summary_path = f"{PATH_RESULTS}/summary.json"
    spark.createDataFrame([(json.dumps(all_summary, indent=2, ensure_ascii=False,
                                         default=str),)], ["summary"]).coalesce(1) \
        .write.mode("overwrite").text(summary_path)
    print(f"\nSummary: {summary_path}")
except Exception as e:
    print(f"Save err: {e}")

impressions.unpersist()
participants_sampled.unpersist()
if mcats_df is not None:
    mcats_df.unpersist()

print("Готово.")